# SHIELD — Missing financial metrics: frozen Random Forest stress test

**Research question:** When an SME provides only 19, 18, 15, or 10 of the 20 required metrics,
how much do the existing model's detection performance and individual risk estimates change?

We load **the existing prototype RF**, preserve all 20 feature columns, and hide observed values with
`NaN`. We compare native RF handling with training-median imputation. **No model is retrained.**
The companies, labels, model and thresholds stay the same; the missing inputs and handling method change.

**Scope:** This measures sensitivity on the historical dataset, not eligibility for financing or deployment
readiness. The historical test split has already been used in development, so results are exploratory.
It does not establish accuracy for Malaysian SMEs, startups, or a different accounting period.
Risk probabilities from SMOTE training are not automatically calibrated real-world default probabilities.
The existing application's completeness gate is unchanged.


## 1. Install the model-compatible environment

Use a **CPU runtime**; a GPU is unnecessary. These core package versions match the saved RF and backend.
Use Python 3.10–3.12 for these pins. If Colab asks you to restart the runtime, restart and continue from
section 2. If section 4 reports a version mismatch, restart the session after installation.

Only load your own trusted `.joblib` artifact: loading it executes Python object deserialization.


In [ ]:
%pip install -q numpy==1.26.4 pandas==2.2.2 scikit-learn==1.6.1 imbalanced-learn==0.12.4 joblib==1.4.2 matplotlib==3.9.4


## 2. Select the project folder

Open this notebook in Colab. Put the following files in Google Drive, preserving these folder names.
The default folder is `/content/drive/MyDrive/SHIELD_Project`; change it if yours differs.

```text
SHIELD_Project/
  model/SHIELD_BestModel_RF_8020_CV10.joblib
  data/preprocessed_dataset.csv
  data/X_train_processed_8020.csv
  data/y_train_processed_8020.csv
  data/X_test_processed_8020.csv
  data/y_test_processed_8020.csv
```

You do **not** need to upload any helper Python files. The experiment functions are embedded below.
No new split is made for scoring: the source CSV is used to verify the existing split and recover the
original training companies for medians. File or split mismatches stop the experiment.


In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/SHIELD_Project')
except ImportError:
    PROJECT = Path.cwd()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent

OUTPUT_BASE = PROJECT / 'reports' / 'missing_data'
print('Project:', PROJECT)
print('Results:', OUTPUT_BASE)


## 3. Freeze the experiment settings before viewing the results

- **20 single-metric cases:** each metric is missing for all test companies in turn.
- **Random missing inputs:** exactly 1, 2, 5, 10 or 20 values are hidden per company; identities vary
  between companies. Twenty seeded repetitions use identical masks for both handling methods.
- **Grouped proxies:** missing four-season EPS, all four selected share-based measures, or five
  leverage/solvency measures. These are controlled masks, not observed Malaysian missingness patterns.
- **Methods:** native `NaN` routing; median replacement using only verified original training rows.
- **Primary decision cutoff:** 0.60, matching SHIELD's high-risk flag. A fixed 0.50 reference is also
  reported for comparison with historical model evaluation. Neither cutoff is tuned here.
- **Risk tiers:** Low <0.30; Moderate 0.30–<0.60; High 0.60–<0.80; Critical >=0.80.
- **20 missing:** a no-information control, not a proposed supported input case.

Do not declare a missing-data policy acceptable just because it reproduces an already weak baseline.
Record your intended minimum detection performance and maximum false-alarm rate before reviewing results.
Formal policy selection should use development data followed by independent validation.


In [ ]:
RANDOM_REPEATS = 20
MISSING_COUNTS = (1, 2, 5, 10, 20)
SEED = 42
BOOTSTRAP_DRAWS = 500
print('Predictions:', 1 + 2 * (20 + 3 + RANDOM_REPEATS * len(MISSING_COUNTS)))
print('No RF fitting, cross-validation search, or threshold optimisation.')


## 4. Load the experiment functions

The full tested runner is included in this cell. It verifies the model's exact feature names/order and
the saved labels. Original train/test membership must reproduce the saved numeric seed-42 stratified
80:20 split. Source values are checked within `1e-8` to allow the source CSV's decimal rounding.

The saved training file contains original rows followed by SMOTE-generated rows. Medians use only
the verified original prefix. SMOTE is never applied to test records or at prediction time.

**Why keep 20 columns?** Removing one column violates the saved model's schema. Keeping the column with
`NaN` tests absent information. When training did not include missing values, native RF routing uses
the child with more training samples; a successful prediction is not proof of robustness.


In [ ]:
"""Frozen-model stress test for SHIELD's saved 20-feature Random Forest.

This runner is embedded in the Colab notebook; no helper upload is required.
It never trains a model, tunes a threshold, or writes production artifacts.
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

MODEL_FILE = "SHIELD_BestModel_RF_8020_CV10.joblib"
DATA_FILES = ["preprocessed_dataset.csv", "X_train_processed_8020.csv",
              "y_train_processed_8020.csv", "X_test_processed_8020.csv",
              "y_test_processed_8020.csv"]
METHODS = ("native_nan", "training_median")
THRESHOLDS = (0.60, 0.50)  # App high-risk cutoff; historical classification reference.
TIER_EDGES = np.array([0.30, 0.60, 0.80])
GROUPS = {
    "earnings_history_missing": ["Persistent EPS in the Last Four Seasons"],
    "share_based_unavailable": ["Persistent EPS in the Last Four Seasons",
                               "Per Share Net profit before tax",
                               "Net Value Per Share (A)", "Net Value Per Share (B)"],
    "leverage_records_missing": ["Debt ratio %", "Net worth/Assets",
                                 "Borrowing dependency", "Degree of Financial Leverage (DFL)",
                                 "Retained Earnings to Total Assets"],
}


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_inputs(project):
    """Recover original train rows and verify their match to the saved split."""
    project = Path(project)
    model_path = project / "model" / MODEL_FILE
    paths = {name: project / "data" / name for name in DATA_FILES}
    for path in [model_path, *paths.values()]:
        if not path.is_file():
            raise FileNotFoundError(f"Required file missing: {path}")
    if importlib.metadata.version("scikit-learn") != "1.6.1":
        raise RuntimeError("Use scikit-learn==1.6.1, matching the saved model. Run the notebook installation cell and restart the runtime if necessary.")
    with warnings.catch_warnings():
        warnings.simplefilter("error", InconsistentVersionWarning)
        model = joblib.load(model_path)
    steps = [(name, type(step).__name__) for name, step in model.steps]
    if steps != [("smote", "SMOTE"), ("model", "RandomForestClassifier")]:
        raise ValueError(f"Unexpected saved pipeline: {steps}. This experiment targets the current prototype RF.")
    features = list(model.feature_names_in_)
    if len(features) != 20 or list(model.classes_) != [0, 1]:
        raise ValueError("Expected 20 features and class order [0, 1].")

    train = pd.read_csv(paths["X_train_processed_8020.csv"])
    y_train = pd.read_csv(paths["y_train_processed_8020.csv"]).iloc[:, 0]
    test = pd.read_csv(paths["X_test_processed_8020.csv"])
    y_test = pd.read_csv(paths["y_test_processed_8020.csv"]).iloc[:, 0]
    raw = pd.read_csv(paths["preprocessed_dataset.csv"])
    for name, X, y in [("saved train", train, y_train), ("saved test", test, y_test)]:
        if list(X.columns) != features or len(X) != len(y):
            raise ValueError(f"{name}: feature order or feature/label alignment differs.")
        if not np.isfinite(X.to_numpy(dtype=float)).all() or set(y.unique()) != {0, 1}:
            raise ValueError(f"{name}: expected complete numeric inputs and both binary classes.")
        if ((X < 0) | (X > 1)).any().any():
            raise ValueError(f"{name}: expected normalised values in [0, 1].")
    if raw.columns.duplicated().any() or "Bankrupt" not in raw or not set(features).issubset(raw.columns):
        raise ValueError("Original source CSV must contain Bankrupt and all 20 feature names.")
    if set(raw.Bankrupt.unique()) != {0, 1}:
        raise ValueError("Original target must be numeric 0/1.")

    # The numeric stratified seed-42 split reproduces the saved files. Fail rather
    # than silently fitting medians on synthetic records or mismatched companies.
    original_train, original_test, original_y_train, original_y_test = train_test_split(
        raw[features], raw.Bankrupt, test_size=0.20, random_state=42, stratify=raw.Bankrupt,
    )
    n_original = len(original_train)
    if len(train) < n_original or len(test) != len(original_test):
        raise ValueError("Saved files do not match the original 80:20 partition sizes.")
    np.testing.assert_allclose(original_train.to_numpy(), train.iloc[:n_original].to_numpy(), rtol=0, atol=1e-8)
    np.testing.assert_allclose(original_test.to_numpy(), test.to_numpy(), rtol=0, atol=1e-8)
    np.testing.assert_array_equal(original_y_train.to_numpy(), y_train.iloc[:n_original].to_numpy())
    np.testing.assert_array_equal(original_y_test.to_numpy(), y_test.to_numpy())
    if set(original_train.index) & set(original_test.index):
        raise AssertionError("Train/test source rows overlap.")
    # Use the full-precision ORIGINAL prefix verified against source, not SMOTE rows.
    medians = train.iloc[:n_original].median()
    if not np.isfinite(medians).all():
        raise ValueError("Training medians are not finite.")
    info = {
        "model_sha256": file_hash(model_path),
        "data_sha256": {name: file_hash(path) for name, path in paths.items()},
        "pipeline_steps": steps, "feature_order": features,
        "original_training_rows": n_original, "synthetic_training_rows": len(train) - n_original,
        "test_rows": len(test), "test_bankrupt": int(y_test.sum()),
        "median_source": "verified original training prefix; synthetic SMOTE rows excluded",
        "partition_verification": "source seed-42 stratified 80:20 split matches saved features within 1e-8 and labels exactly",
    }
    return model, test, y_test.to_numpy(dtype=int), medians, original_test.index.to_numpy(), info


def random_mask(n_rows, n_features, count, seed):
    if not 0 <= count <= n_features:
        raise ValueError("Missing-feature count must be between 0 and feature count.")
    rng = np.random.default_rng(seed)
    positions = np.argsort(rng.random((n_rows, n_features)), axis=1)[:, :count]
    mask = np.zeros((n_rows, n_features), dtype=bool)
    mask[np.arange(n_rows)[:, None], positions] = True
    return mask


def apply_mask(X, mask, method, medians):
    if mask.shape != X.shape or mask.dtype != bool:
        raise ValueError("Mask shape/type differs from the financial matrix.")
    missing = X.mask(mask)
    if method == "native_nan":
        return missing
    if method == "training_median":
        return missing.fillna(medians)
    raise ValueError(f"Unknown missing-data method: {method}")


def score(model, X):
    p = np.asarray(model.predict_proba(X)[:, 1], dtype=float)
    if p.shape != (len(X),) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Model returned invalid probabilities.")
    return p


def metrics(y, p, reference, threshold):
    prediction, baseline_prediction = p >= threshold, reference >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    movement = np.abs(p - reference)
    return {
        "threshold": threshold, "accuracy": accuracy_score(y, prediction),
        "precision": precision_score(y, prediction, zero_division=0),
        "recall": recall_score(y, prediction, zero_division=0),
        "f1": f1_score(y, prediction, zero_division=0),
        "average_precision": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p), "false_positive_rate": fp / (tn + fp),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "mean_probability_change_pp": float(movement.mean() * 100),
        "p95_probability_change_pp": float(np.quantile(movement, 0.95) * 100),
        "max_probability_change_pp": float(movement.max() * 100),
        "bankrupt_probability_change_pp": float(movement[y == 1].mean() * 100),
        "tier_change_pct": float(np.mean(np.digitize(p, TIER_EDGES) != np.digitize(reference, TIER_EDGES)) * 100),
        "decision_change_pct": float(np.mean(prediction != baseline_prediction) * 100),
        "newly_missed_bankrupt": int(np.sum((y == 1) & baseline_prediction & ~prediction)),
        "newly_detected_bankrupt": int(np.sum((y == 1) & ~baseline_prediction & prediction)),
        "unique_probabilities": int(len(np.unique(p))),
    }


def baseline_bootstrap(y, p, repeats=500, seed=7241):
    """Stratified company resampling; conditional on this reused test set."""
    if repeats < 2:
        raise ValueError("Use at least two bootstrap draws.")
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(y == label) for label in [0, 1]]
    values = {name: [] for name in ["recall", "precision", "average_precision", "roc_auc", "brier"]}
    for _ in range(repeats):
        rows = np.concatenate([rng.choice(indices, len(indices), replace=True) for indices in groups])
        resampled_y, resampled_p = y[rows], p[rows]
        values["recall"].append(recall_score(resampled_y, resampled_p >= 0.60, zero_division=0))
        values["precision"].append(precision_score(resampled_y, resampled_p >= 0.60, zero_division=0))
        values["average_precision"].append(average_precision_score(resampled_y, resampled_p))
        values["roc_auc"].append(roc_auc_score(resampled_y, resampled_p))
        values["brier"].append(brier_score_loss(resampled_y, resampled_p))
    return pd.DataFrame([{"metric": name, "lower_95": np.quantile(v, .025),
                          "upper_95": np.quantile(v, .975), "bootstrap_draws": repeats}
                         for name, v in values.items()])


def run_experiment(project, output_base=None, repeats=20, seed=42,
                   missing_counts=(1, 2, 5, 10, 20), bootstrap_draws=500):
    if repeats < 2 or not missing_counts or len(set(missing_counts)) != len(missing_counts):
        raise ValueError("Use >=2 repeats and distinct missing-feature counts.")
    if any(not 1 <= count <= 20 for count in missing_counts):
        raise ValueError("Counts must be between 1 and 20.")
    model, X, y, medians, source_rows, info = load_inputs(project)
    packages = {name: importlib.metadata.version(name) for name in
                ["scikit-learn", "imbalanced-learn", "numpy", "pandas", "joblib", "scipy"]}
    config = {"experiment_version": "shield-missing-data-v1", "seed": seed,
              "random_repeats": repeats, "missing_counts": list(missing_counts),
              "methods": METHODS, "thresholds": THRESHOLDS, "tier_edges": TIER_EDGES.tolist(),
              "bootstrap_draws": bootstrap_draws, "groups": GROUPS, "packages": packages,
              "test_usage": "historical test set reused in development; exploratory stress test",
              "scope": "frozen RF predictions only; no SHAP/Groq/API validation; no Malaysian startup validation",
              **info}
    run_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]
    out = Path(output_base or Path(project) / "reports" / "missing_data") / run_id
    out.mkdir(parents=True, exist_ok=True)
    baseline = score(model, X)
    rows, prediction_records, mask_records = [], [], []
    baseline_metrics = {threshold: metrics(y, baseline, baseline, threshold) for threshold in THRESHOLDS}
    for threshold in THRESHOLDS:
        rows.append({"family": "baseline", "scenario": "complete_20", "method": "complete",
                     "repeat": 0, "missing_count": 0, **baseline_metrics[threshold]})
    print(f"Verified original train={info['original_training_rows']}, synthetic={info['synthetic_training_rows']}; test={len(X)}, bankrupt={int(y.sum())}", flush=True)
    print("Model stays frozen. Primary threshold=0.60; secondary reference=0.50.", flush=True)

    def evaluate(family, scenario, mask, repeat=0):
        count = int(mask.sum(axis=1)[0])
        assert np.all(mask.sum(axis=1) == count)
        mask_records.append({"family": family, "scenario": scenario, "repeat": repeat,
                             "missing_count": count, "mask_sha256": hashlib.sha256(mask.tobytes()).hexdigest()})
        for method in METHODS:
            modified = apply_mask(X, mask, method, medians)
            p = score(model, modified)
            for threshold in THRESHOLDS:
                result = metrics(y, p, baseline, threshold)
                for name in ["recall", "precision", "average_precision", "roc_auc", "accuracy", "false_positive_rate"]:
                    result[name + "_change_pp"] = (result[name] - baseline_metrics[threshold][name]) * 100
                result["brier_change"] = result["brier"] - baseline_metrics[threshold]["brier"]
                rows.append({"family": family, "scenario": scenario, "method": method,
                             "repeat": repeat, "missing_count": count, **result})
            prediction_records.append(pd.DataFrame({
                "test_row": np.arange(len(X)), "source_row": source_rows, "true_label": y,
                "family": family, "scenario": scenario, "method": method, "repeat": repeat,
                "missing_count": count, "complete_probability": baseline, "probability": p,
                "missing_features": [" | ".join(np.asarray(X.columns)[row]) for row in mask],
            }))

    for i, feature in enumerate(X.columns):
        mask = np.zeros(X.shape, dtype=bool)
        mask[:, i] = True
        evaluate("single", feature.strip(), mask)
    lookup = {name.strip(): i for i, name in enumerate(X.columns)}
    for scenario, features in GROUPS.items():
        mask = np.zeros(X.shape, dtype=bool)
        mask[:, [lookup[feature] for feature in features]] = True
        evaluate("group", scenario, mask)
    for count in missing_counts:
        for repeat in range(repeats):
            # A repeat supplies one mask shared by both methods, independent of y.
            mask_seed = int(np.random.SeedSequence([seed, count, repeat]).generate_state(1)[0])
            evaluate("random", f"random_{count}_missing", random_mask(len(X), X.shape[1], count, mask_seed), repeat)
        print(f"Finished {count} missing metrics/company, {repeats} paired repetitions.", flush=True)

    results = pd.DataFrame(rows)
    results.to_csv(out / "scenario_metrics.csv", index=False)
    pd.concat(prediction_records, ignore_index=True).to_csv(out / "company_predictions.csv.gz", index=False, compression="gzip")
    pd.DataFrame(mask_records).to_csv(out / "mask_manifest.csv", index=False)
    pd.DataFrame({"feature": medians.index, "original_training_median": medians.values}).to_csv(out / "training_medians.csv", index=False)
    pd.DataFrame({"test_row": np.arange(len(X)), "source_row": source_rows, "true_label": y,
                  "complete_probability": baseline}).to_csv(out / "complete_predictions.csv", index=False)
    baseline_bootstrap(y, baseline, bootstrap_draws).to_csv(out / "baseline_bootstrap.csv", index=False)
    summary_metrics = ["recall", "precision", "average_precision", "roc_auc", "brier", "false_positive_rate",
                       "tier_change_pct", "decision_change_pct", "mean_probability_change_pp", "newly_missed_bankrupt"]
    random = results[results.family == "random"]
    summary = random.groupby(["missing_count", "method", "threshold"])[summary_metrics].agg(["mean", "std", "min", "max"])
    summary.columns = ["_".join(column) for column in summary.columns]
    summary.reset_index().to_csv(out / "random_summary.csv", index=False)
    config["run_id"] = run_id
    (out / "run_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    write_readout(results, config, out)
    print(f"Saved {len(results)} metric rows to {out}", flush=True)
    return {"output_dir": out, "results": results, "config": config}


def write_readout(results, config, out):
    primary = results[results.threshold == .60]
    baseline = primary[primary.family == "baseline"].iloc[0]
    lines = ["SHIELD MISSING-DATA STRESS TEST", "",
             "Exploratory evaluation on the historical test partition; no retraining or production policy changes.",
             f"Test companies: {config['test_rows']}; bankrupt: {config['test_bankrupt']}.",
             "Primary high-risk threshold is 0.60. Average precision summarises the precision-recall curve.",
             f"Complete-input recall: {baseline.recall:.3f}; precision: {baseline.precision:.3f}; AP: {baseline.average_precision:.3f}; missed bankrupt: {int(baseline.fn)}.", ""]
    for method in METHODS:
        worst = primary[(primary.family == "single") & (primary.method == method)].sort_values(["recall", "average_precision"]).iloc[0]
        lines.append(f"{method}: lowest single-missing recall is {worst.recall:.3f} when '{worst.scenario}' is missing; AP={worst.average_precision:.3f}; newly missed bankrupt={int(worst.newly_missed_bankrupt)}.")
    lines.extend(["", "Read scenario_metrics.csv for every metric and paired change from baseline.",
                  "Random repetition spread measures variation in simulated masks, not a confidence interval across independent companies.",
                  "baseline_bootstrap.csv gives conditional stratified company-bootstrap intervals; it does not correct historical test reuse.",
                  f"With {config['test_bankrupt']} bankrupt companies, one changed detection alters recall by about {100 / config['test_bankrupt']:.2f} percentage points.",
                  "The 20-missing scenario is a no-information control; a returned number is not a useful individual assessment.",
                  "Share-based/historical group masks are proxies. Structurally undefined metrics must not be treated as observed financial values.",
                  "Brier score is descriptive; probabilities from SMOTE training are not automatically calibrated real-world default probabilities.",
                  "No result automatically approves a 19/20 gate. Define acceptable absolute detection/false-alarm performance and degradation before independent validation.",
                  "Malaysia/new-business suitability, missingness related to distress, and SHAP/advisory behaviour need separate validation."])
    (out / "READOUT.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")


def plot_results(output_dir, show=True):
    import matplotlib.pyplot as plt
    out = Path(output_dir)
    results = pd.read_csv(out / "scenario_metrics.csv")
    primary = results[results.threshold == .60]
    baseline = primary[primary.family == "baseline"].iloc[0]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for method in METHODS:
        grouped = primary[(primary.family == "random") & (primary.method == method)].groupby("missing_count")
        for axis, metric, label in zip(axes, ["recall", "average_precision", "false_positive_rate", "tier_change_pct"],
                                       ["Bankruptcy recall", "Average precision", "False-positive rate", "Companies changing risk tier (%)"]):
            mean, low, high = grouped[metric].mean(), grouped[metric].min(), grouped[metric].max()
            axis.plot(mean.index, mean.values, marker="o", label=method)
            axis.fill_between(mean.index, low.values, high.values, alpha=.15)
            axis.axhline(baseline[metric], color="black", linestyle="--", linewidth=1)
            axis.set(xlabel="Missing metrics per company", ylabel=label, xticks=mean.index)
            axis.grid(alpha=.2)
    axes[0].legend(fontsize=8)
    fig.suptitle("Frozen RF: random missing metrics (bands = min/max across masks, not confidence intervals)", fontsize=11)
    fig.tight_layout()
    fig.savefig(out / "random_degradation.png", dpi=160, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)

    single = primary[primary.family == "single"].pivot(index="scenario", columns="method", values="recall_change_pp")
    single = single.loc[single.min(axis=1).sort_values().index]
    fig, axis = plt.subplots(figsize=(11, 9))
    single.plot.barh(ax=axis)
    axis.axvline(0, color="black", linewidth=1)
    axis.set(xlabel="Bankruptcy recall change from complete inputs (percentage points)", ylabel="Missing metric",
             title="19/20 inputs: the identity of the missing metric matters")
    axis.grid(axis="x", alpha=.2)
    fig.tight_layout()
    fig.savefig(out / "single_feature_effects.png", dpi=160, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)



## 5. Verify inputs and demonstrate 19 columns versus 19 observed values

The first case below should produce a feature-schema error. The second keeps 20 columns with one `NaN`
and should produce a finite score. This is a technical smoke check on one company, not an accuracy test.


In [ ]:
model, X_test, y_test, training_medians, source_rows, input_info = load_inputs(PROJECT)
display(pd.Series(input_info, name='Input verification'))
print('Test labels:', dict(zip(*np.unique(y_test, return_counts=True))))
try:
    score(model, X_test.iloc[[0]].drop(columns=[X_test.columns[0]]))
except ValueError as exc:
    print('19-column input rejected:', str(exc).splitlines()[0])
else:
    raise AssertionError('Unexpected: model accepted a removed feature column.')

example = X_test.iloc[[0]].copy()
complete_score = score(model, example)[0]
example.iloc[0, 0] = np.nan
missing_score = score(model, example)[0]
print(f'First company: complete={complete_score:.3f}; one NaN={missing_score:.3f}')
print('Both methods will now be compared across ALL test companies, not just demo profiles.')


## 6. Run the paired stress test

Default settings perform **247 prediction passes**, not 247 model trainings. Each pass scores the
same test companies. Results are written to a folder identified by model/data hashes, package versions
and settings. Identical inputs/settings reproduce the same folder; changed inputs/settings create a new one.
Each scenario appears at both fixed decision cutoffs in `scenario_metrics.csv`.

The same observed entries are kept for both methods. A recorded zero remains zero. Original input files
and the saved model are never overwritten. The median method is an experimental inference policy for this
existing RF; it is not a newly trained imputation-aware model.


In [ ]:
RUN = run_experiment(
    PROJECT, output_base=OUTPUT_BASE, repeats=RANDOM_REPEATS,
    seed=SEED, missing_counts=MISSING_COUNTS, bootstrap_draws=BOOTSTRAP_DRAWS,
)
RESULTS = RUN['results']
OUT = RUN['output_dir']
print((OUT / 'READOUT.txt').read_text())


## 7. Read the complete-input baseline first

- **Recall:** fraction of labelled bankrupt companies detected. False negatives are missed bankrupt firms.
- **Precision:** fraction of flagged companies that are labelled bankrupt.
- **False-positive rate:** fraction of non-bankrupt companies incorrectly flagged.
- **Average precision (AP):** summarises the precision-recall curve; compare against the bankruptcy
  prevalence printed below. This is AP, not trapezoidal PR-AUC.
- **ROC-AUC:** ranking quality; interpret alongside AP for this heavily imbalanced dataset.
- **Brier score:** squared probability error; lower is better. It does not by itself prove calibration.
- **Accuracy:** secondary, because correctly labelling most non-bankrupt firms can hide weak recall.

There are only **44 bankrupt test companies** in the current saved split. One changed detection moves
recall by about **2.27 percentage points**. Bootstrap intervals are conditional on this test set and its
class balance; they do not correct prior test reuse or domain mismatch.


In [ ]:
DISPLAY_METRICS = ['threshold', 'accuracy', 'precision', 'recall', 'average_precision',
                   'roc_auc', 'brier', 'false_positive_rate', 'tp', 'fn', 'fp', 'tn']
display(RESULTS[RESULTS.family == 'baseline'][DISPLAY_METRICS].round(4))
print(f'Bankruptcy prevalence / constant-score AP reference: {y_test.mean():.4f}')
display(pd.read_csv(OUT / 'baseline_bootstrap.csv').round(4))


## 8. Which individual missing metric matters most?

Every company has 19 observed metrics in these tests. Rows differ in WHICH metric is absent.
`recall_change_pp` and `average_precision_change_pp` are percentage-point changes from complete inputs.
`newly_missed_bankrupt` counts firms detected by the baseline but missed after masking; recoveries are
reported separately so improvements cannot hide newly missed firms. Tier changes are disagreements with
the complete-input model, not evidence that either tier is correct.


In [ ]:
PRIMARY = RESULTS[RESULTS.threshold == 0.60]
SINGLE = PRIMARY[PRIMARY.family == 'single'].copy()
columns = ['scenario', 'method', 'recall', 'recall_change_pp', 'precision',
           'average_precision', 'average_precision_change_pp', 'fn', 'newly_missed_bankrupt',
           'newly_detected_bankrupt', 'tier_change_pct', 'mean_probability_change_pp']
display(SINGLE.sort_values(['recall_change_pp', 'average_precision_change_pp'])[columns].round(4))


## 9. How does performance change as more inputs disappear?

Random masks are independent of the true class, so they are a controlled random-missing experiment.
Real missingness may relate to distress, bookkeeping practices, business age or legal structure.
Twenty masks on the same companies are **not twenty independent datasets**. Their min/max and standard
deviation describe sensitivity to mask choice, not a confidence interval for population accuracy.
Recall can rise simply because more companies are flagged. Check the false-positive rate and AP
alongside recall; the all-missing control makes this failure mode visible.


In [ ]:
random_summary = pd.read_csv(OUT / 'random_summary.csv')
display(random_summary[random_summary.threshold == .60][[
    'missing_count', 'method', 'recall_mean', 'recall_min', 'recall_max',
    'average_precision_mean', 'brier_mean', 'false_positive_rate_mean', 'tier_change_pct_mean',
    'mean_probability_change_pp_mean', 'newly_missed_bankrupt_mean',
]].round(4))
plot_results(OUT)


## 10. New-business and missing-record proxies

These grouped cases remove related information together. They do **not** simulate a complete startup
population. A truly undefined share-based metric or nonexistent four-quarter history is different from
an unreported, otherwise meaningful value. Median imputation in this notebook is only a stress-test
comparison and does not establish that imputing structurally undefined ratios is suitable.


In [ ]:
display(PRIMARY[PRIMARY.family == 'group'][[
    'scenario', 'missing_count', 'method', 'recall', 'recall_change_pp',
    'precision', 'average_precision', 'fn', 'fp', 'newly_missed_bankrupt',
    'tier_change_pct', 'mean_probability_change_pp',
]].round(4))


## 11. Inspect individual companies and export the evidence

The detailed output uses numeric source-row IDs, not company names. Pick a method/scenario below to see
which risk estimates moved the most. Do not interpret the changed score alone as model accuracy; use the
true label and full-set metrics. The compressed CSV contains all scenario/company predictions and the
exact names of the hidden metrics, so a result can be reproduced and explained.

Outputs: metrics for all scenarios, random-repeat summaries, complete-input predictions, individual
predictions, original-training medians, mask hashes, baseline bootstrap intervals, figures, run metadata,
and a plain-language readout. The ZIP excludes the saved model and original source data.


In [ ]:
predictions = pd.read_csv(OUT / 'company_predictions.csv.gz')
inspection = predictions[(predictions.family == 'group') &
                         (predictions.scenario == 'share_based_unavailable') &
                         (predictions.method == 'native_nan')].copy()
inspection['change_pp'] = (inspection.probability - inspection.complete_probability) * 100
display(inspection.loc[inspection.change_pp.abs().sort_values(ascending=False).index].head(15))

import shutil
zip_path = shutil.make_archive(str(OUT.parent / f'SHIELD_missing_data_{RUN["config"]["run_id"]}'),
                               'zip', root_dir=OUT)
print('Results ZIP:', zip_path)
# Download when you are ready in Colab:
# from google.colab import files
# files.download(zip_path)


## 12. What can we conclude, and what comes next?

Write down the baseline first, then the largest single-feature damage, the random-missing trend,
the grouped-pattern damage, and how the two handling methods differ. Report exact missed-bankrupt and
false-alarm counts, not just a high accuracy percentage. A method can preserve recall while sharply
increasing false alarms or shifting risk tiers.

This notebook answers **how the frozen RF reacts**; it does not automatically approve incomplete profiles.
Use the following next steps:

1. Predefine acceptable absolute recall/false-alarm performance and allowable deterioration with your
   supervisor. If the complete-input baseline is inadequate, missing-data tolerance does not fix it.
2. Treat these results as exploratory. Select handling policies using training/development data, then
   validate the frozen policy on genuinely fresh companies; repeated historical test reuse limits claims.
3. If necessary, separately train an imputation-aware or missingness-aware model, fitting preprocessing
   inside training folds. That is a new modelling experiment, not something this notebook does.
4. Validate on representative Malaysian SMEs, including startup/structure-specific patterns. Artificially
   hiding values in mature-company records does not establish startup suitability.
5. Before enabling incomplete inputs, test the full API, SHAP, advisory text and report path. Explanations
   must identify missing/imputed metrics and must not present imputed values as observed financial facts.

**Retain the current Insufficient Data gate until a justified policy has been validated.**

### Technical references

- [RandomForestClassifier 1.6: native missing-value routing](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [scikit-learn: leakage and preprocessing using training data only](https://scikit-learn.org/stable/common_pitfalls.html)
- [imbalanced-learn Pipeline: sampling during fit](https://imbalanced-learn.org/stable/references/generated/imblearn.pipeline.Pipeline.html)
- [scikit-learn: average precision](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.average_precision_score.html)
